# EXP-036 · Caption-supervised visual adaptation

Attach the private `exp036_inputs.zip` dataset. Enable a GPU (T4 ×2 uses GPU 0) and Internet
for dependencies and the pinned public SigLIP2 download. Set `AUTHORIZE_RUN = True` then
**Run All**. The run uses official training captions only as auxiliary targets, then selects
the final checkpoint and blend on Development.

Optional mounts are the exact pinned SigLIP2 files and a prior results or progress archive.
Keep the notebook and inputs private. Results are for manual review and upload.


In [ ]:
AUTHORIZE_RUN = False
MODEL_ROOT = ""  # Optional exact local Kaggle model mount.
RESUME = ""      # Optional mounted progress/results ZIP or extracted directory.
SESSION_HOURS = 10.5
MANIFEST_SHA = 'ff34edd58a442082b6848f77dabcc4e2804582d15350d73e033de2caf7c2b4d2'
ARCHIVE_SHA = '3e59a91ea6e4c9cdda0ecc3164a2a557d6425cb5ddc4e0e868bac1d6128ecddc'


In [ ]:
import hashlib
import json
import subprocess
import sys
import time
from pathlib import Path
from zipfile import ZipFile

if AUTHORIZE_RUN is not True:
    raise RuntimeError("Attach the private EXP-036 input dataset and enable GPU/Internet before setting AUTHORIZE_RUN = True.")
started = time.time()
mount = Path("/kaggle/input")
out = Path("/kaggle/working")
if not mount.is_dir() or not out.is_dir():
    raise RuntimeError("Run this notebook in Kaggle.")

def sha(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

try:
    matches = [p for p in mount.rglob("manifest.json") if sha(p) == MANIFEST_SHA]
    if len(matches) == 1:
        inputs = matches[0].parent
    elif not matches:
        zips = list(mount.rglob("exp036_inputs.zip"))
        if len(zips) != 1 or sha(zips[0]) != ARCHIVE_SHA:
            raise ValueError("Attach the exact exp036_inputs.zip dataset or its extracted directory.")
        scratch = Path("/kaggle/temp/exp036_input")
        with ZipFile(zips[0]) as zipped:
            zipped.extractall(scratch)  # This path is reached only for the exact pinned ZIP.
        inputs = scratch / "exp036_inputs"
    else:
        raise ValueError("Attach exactly one copy of the EXP-036 handoff.")
    if sha(inputs / "manifest.json") != MANIFEST_SHA:
        raise ValueError("Input manifest SHA-256 mismatch.")
    m = json.loads((inputs / "manifest.json").read_text())
    for name, digest in m["files"].items():
        if not (inputs / name).is_file() or sha(inputs / name) != digest:
            raise ValueError(f"Missing or corrupt pinned input {name}")
    launcher = inputs / "runtime/scripts/exp036_kaggle.py"
    command = [sys.executable, "-u", str(launcher), "--inputs", str(inputs), "--handoff", MANIFEST_SHA,
               "--started", str(started), "--hours", str(SESSION_HOURS)]
    if MODEL_ROOT:
        command += ["--model", MODEL_ROOT]
    if RESUME:
        command += ["--resume", RESUME]
    subprocess.run(command, check=True)
except Exception as error:
    diagnostic = out / "exp036_bootstrap_error.json"
    diagnostic.write_text(json.dumps({"state": "BOOTSTRAP_FAILED", "reason": str(error)}, indent=2))
    print(diagnostic.read_text(), flush=True)


## Outputs

The Output panel holds the sealed results and separate submissions. Save a **private** version
with outputs or download the files before retiring the session. A results archive with
`RESUMABLE_STOP` or a periodic progress archive can be mounted in a fresh session with the
same GPU model. Fill in `RESUME`; training continues at the sealed optimizer boundary.
An abruptly terminated session can lose work that Kaggle has not saved or you have not downloaded.


In [ ]:
from IPython.display import FileLink, display
from pathlib import Path

working = Path("/kaggle/working")
run = working / "exp036_run"
for path in (run / "status.json", run / "preflight.json", working / "exp036_diagnostic.json"):
    if path.is_file():
        print(path.read_text())
for path in (working / "exp036_results.zip", working / "exp036_results.zip.sha256",
             working / "exp036_progress.zip", working / "exp036_diagnostic.zip",
             run / "exp036_standalone_submission.zip", run / "exp036_ensemble_submission.zip"):
    if path.is_file():
        display(FileLink(str(path)))
if (run / "metrics.json").is_file():
    report = json.loads((run / "metrics.json").read_text())
    print("Selected epoch", report["selected_epoch"], "; selected ensemble alpha", report["ensemble"]["alpha"])
    for name, row in (("Corrected Ridge", report["reference"]), ("CS-CVA", report["standalone"]),
                      ("Ensemble", report["ensemble"]["metrics"])):
        print(name, row["cultural_contrast"], row["log_loss"])
